# Notebook 04 — 전체 시스템 통합 테스트

**목표:** 전체 파이프라인(데이터 생성 → 모델 학습 → API 서빙 → 대시보드)이 실제로 동작하는지 종합 검증한다.

**검증 항목:**
1. 데이터 생성 → 모델 재학습 API
2. 다양한 이상 패턴 탐지 정확도
3. API 응답 속도 (단건 / 배치)
4. 시스템 전체 흐름 E2E 시나리오

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import time
import json
import numpy as np
import pandas as pd
import requests
from datetime import datetime

API_BASE = 'http://localhost:8000'
print('통합 테스트 시작:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

## 1. 헬스체크 및 모델 상태 확인

In [ ]:
health = requests.get(f'{API_BASE}/health').json()
model_info = requests.get(f'{API_BASE}/model/info').json()

print(f'서버 상태: {health["status"]}')
print(f'모델 학습 여부: {model_info["is_fitted"]}')
print(f'모델 유형: {model_info["model_type"]}')
print(f'contamination: {model_info["contamination"]}')

## 2. 모델 재학습 (최신 데이터 반영)

In [ ]:
print('모델 재학습 시작...')
start = time.time()
train_result = requests.post(f'{API_BASE}/model/train?contamination=0.05').json()
elapsed = time.time() - start

print(f'학습 완료 ({elapsed:.2f}초)')
print(f'  학습 샘플: {train_result["n_samples"]:,}건')
print(f'  Precision: {train_result["precision"]}')
print(f'  Recall:    {train_result["recall"]}')
print(f'  F1:        {train_result["f1"]}')

## 3. 배치 추론 속도 측정

In [ ]:
rng = np.random.RandomState(42)
N = 100
latencies = []

for i in range(N):
    payload = {
        'sensor_id': f'bench_{i:03d}',
        'temperature': float(rng.normal(70, 5)),
        'vibration':   float(rng.normal(0.5, 0.1)),
        'current':     float(rng.normal(12, 1)),
    }
    t0 = time.time()
    requests.post(f'{API_BASE}/detect', json=payload)
    latencies.append((time.time() - t0) * 1000)

latencies = np.array(latencies)
print(f'=== 추론 속도 벤치마크 (N={N}) ===')
print(f'  평균:   {latencies.mean():.1f} ms')
print(f'  중앙값: {np.median(latencies):.1f} ms')
print(f'  P95:    {np.percentile(latencies, 95):.1f} ms')
print(f'  최대:   {latencies.max():.1f} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(latencies, bins=30, color='#3B82F6', edgecolor='white')
axes[0].axvline(latencies.mean(), color='red', linestyle='--', label=f'평균 {latencies.mean():.1f}ms')
axes[0].set_xlabel('응답시간 (ms)')
axes[0].set_ylabel('빈도')
axes[0].set_title('API 응답시간 분포')
axes[0].legend()

axes[1].plot(latencies, color='#3B82F6', linewidth=0.8, alpha=0.7)
axes[1].axhline(latencies.mean(), color='red', linestyle='--', label=f'평균')
axes[1].set_xlabel('요청 번호')
axes[1].set_ylabel('응답시간 (ms)')
axes[1].set_title('요청별 응답시간 추이')
axes[1].legend()

plt.suptitle('POST /detect 응답시간 벤치마크', fontsize=13)
plt.tight_layout()
plt.savefig('../data/benchmark_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. E2E 시나리오 — 공장 라인 이상 감지

In [ ]:
# 시나리오: 공장 라인 가동 중 모터 과열 발생
# t=0~9: 정상 가동
# t=10~14: 모터 온도 점진적 상승 (예열)
# t=15~19: 이상 범위 진입 (과열)

rng = np.random.RandomState(0)
scenario = []

for t in range(20):
    if t < 10:
        temp = rng.normal(70, 2)
        phase = '정상 가동'
    elif t < 15:
        temp = 70 + (t - 10) * 4 + rng.normal(0, 1)  # 점진적 상승
        phase = '온도 상승'
    else:
        temp = rng.normal(95, 3)
        phase = '과열 이상'

    payload = {
        'sensor_id': 'line1_motor',
        'temperature': round(float(temp), 2),
        'vibration':   round(float(rng.normal(0.5, 0.08)), 3),
        'current':     round(float(rng.normal(12, 0.8)), 2),
    }
    r = requests.post(f'{API_BASE}/detect', json=payload).json()
    scenario.append({'t': t, 'phase': phase, **payload, **{
        'score': r['anomaly_score'],
        'is_anomaly': r['is_anomaly'],
        'severity': r['severity'],
    }})

df_scenario = pd.DataFrame(scenario)
print(df_scenario[['t', 'phase', 'temperature', 'score', 'severity']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

colors_e2e = ['#EF4444' if r else '#3B82F6' for r in df_scenario['is_anomaly']]

# 온도 추이
axes[0].plot(df_scenario['t'], df_scenario['temperature'], color='#CBD5E1', linewidth=1.5)
axes[0].scatter(df_scenario['t'], df_scenario['temperature'], c=colors_e2e, s=60, zorder=5)
axes[0].axhline(85, color='orange', linestyle='--', alpha=0.7, label='경고 임계값 (85°C)')
axes[0].set_ylabel('온도 (°C)')
axes[0].set_title('E2E 시나리오 — 모터 과열 감지')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 이상 점수 추이
axes[1].plot(df_scenario['t'], df_scenario['score'], color='#8B5CF6', linewidth=1.5, marker='o')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--', label='정상/이상 경계')
axes[1].fill_between(df_scenario['t'],
                      df_scenario['score'],
                      0,
                      where=[s < 0 and a for s, a in zip(df_scenario['score'], df_scenario['is_anomaly'])],
                      color='#EF4444', alpha=0.3, label='이상 구간')
axes[1].set_xlabel('시간 (초)')
axes[1].set_ylabel('Anomaly Score')
axes[1].set_title('이상 점수 추이')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 구간 배경색
for ax in axes:
    ax.axvspan(0, 9.5, alpha=0.05, color='blue')
    ax.axvspan(9.5, 14.5, alpha=0.05, color='orange')
    ax.axvspan(14.5, 19, alpha=0.05, color='red')

plt.tight_layout()
plt.savefig('../data/e2e_scenario.png', dpi=150, bbox_inches='tight')
plt.show()
print('파란 배경: 정상 가동 | 주황: 온도 상승 | 빨강: 이상 감지')

## 5. Docker Compose 실행 가이드

전체 시스템(Mosquitto + FastAPI + Streamlit)을 하나의 명령어로 실행:

```bash
docker-compose up --build
```

| 서비스 | URL | 역할 |
|--------|-----|------|
| FastAPI | http://localhost:8000/docs | REST API + WebSocket |
| Streamlit | http://localhost:8501 | 실시간 대시보드 |
| Mosquitto | localhost:1883 | MQTT 브로커 |

### 실행 순서
```bash
# 1. 데이터 생성
python scripts/generate_data.py

# 2. 시스템 실행
docker-compose up --build

# 3. 모델 학습
curl -X POST http://localhost:8000/model/train

# 4. 센서 시뮬레이터 실행
python scripts/mqtt_simulator.py --anomaly

# 5. 대시보드 확인
# http://localhost:8501
```

## 6. 프로젝트 총정리

### 구현된 기능
| 기능 | 구현 방식 | 상태 |
|------|----------|------|
| 이상탐지 모델 | IsolationForest (비지도) | ✅ |
| REST API | FastAPI POST /detect | ✅ |
| 실시간 스트리밍 | WebSocket /ws/stream | ✅ |
| 모델 재학습 | POST /model/train | ✅ |
| 통계 API | GET /stats | ✅ |
| 이상 로그 | SQLite 저장 | ✅ |
| 센서 시뮬레이션 | MQTT pub/sub | ✅ |
| 실시간 대시보드 | Streamlit + Plotly | ✅ |
| 컨테이너 배포 | Docker Compose (3서비스) | ✅ |

### 학습 포인트
- **비지도 이상탐지**: 이상 레이블 없이 정상 데이터만으로 모델 학습 가능
- **실시간 아키텍처**: REST와 WebSocket을 함께 제공하여 다양한 클라이언트 대응
- **IoT 패턴**: MQTT pub/sub으로 실제 산업 현장과 유사한 데이터 흐름 구현
- **서비스 엔지니어링**: 모델 재학습 API, 통계 API 등 운영 관점의 기능 포함